# Mimicry Diagnostic — Is a Bigger Model Worth It?

**Status:** COMPLETED. This diagnostic was run with GPT-2 Small as the sensor model.
Results showed GPT-2 Medium (layer 23) achieved AUC=0.967 on the hard mimicry test
vs GPT-2 Small's AUC=0.367. Based on these results, IRIS was upgraded to GPT-2 Large.
See `archive/gpt2_small/notebooks/20_mimicry_diagnostic_results.ipynb` for the original outputs.

**Note:** The code below will NOT reproduce the original comparison because
`load_model()` now loads GPT-2 Large (the upgrade this diagnostic recommended).
This notebook is preserved for documentation purposes.

---

**Original purpose:** Determine whether upgrading the SAE sensor model from GPT-2 Small (124M) to a larger model would help with the 100% mimicry evasion rate.

**Tests:**
1. Does GPT-2 Small's raw activation space separate mimicry from normal at all?
2. Does TF-IDF catch mimicry?
3. GPT-2 Medium (355M) head-to-head comparison
4. Quick SAE feasibility on GPT-2 Medium

**Result:** Upgrade recommended and implemented (GPT-2 Large, 774M, 36 layers, d_model=1280).

*Nathan Cheung | York University | CSSD 2221 | Winter 2026*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/iris')

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_val_score

from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

DRIVE_ROOT = Path('/content/drive/MyDrive/iris')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 0: Prepare Test Data

We need three groups:
- **Mimicry attacks** — educational-sounding injection questions (label=1)
- **Normal prompts** — genuinely benign questions (label=0)
- **Matched controls** — benign educational questions about non-injection topics (label=0)

The matched controls are critical: if a model can't separate 'Explain how to override a system prompt' from 'Explain how gradient descent works', the problem is genuinely hard.

In [ ]:
from src.analysis.adversarial import generate_evasion_prompts
from src.data.dataset import IrisDataset, SYSTEM_PROMPT_TEMPLATE

# Load dataset for normal examples
ds_path = DRIVE_ROOT / 'data' / 'processed' / 'iris_dataset_balanced.json'
expanded_path = DRIVE_ROOT / 'data' / 'processed' / 'iris_dataset_expanded.json'
if expanded_path.exists():
    ds_path = expanded_path
dataset = IrisDataset.load(ds_path)
labels = np.array(dataset.labels)

# Get mimicry attacks
evasion_prompts = generate_evasion_prompts(n=50, seed=42)
mimicry_prompts = [p for p in evasion_prompts if p['evasion_strategy'] == 'mimicry']
mimicry_texts = [p['text'] for p in mimicry_prompts]

# Get normal prompts from dataset
normal_idx = np.where(labels == 0)[0]
np.random.seed(42)
sample_normal_idx = np.random.choice(normal_idx, size=50, replace=False)
normal_texts = [dataset.texts[i] for i in sample_normal_idx]

# Generate matched controls — educational questions about benign topics
# These use the SAME templates as mimicry but with harmless fillers
benign_fillers = [
    "implement gradient descent in a neural network",
    "fine-tune a pre-trained language model on custom data",
    "tokenize text for transformer models",
    "evaluate model performance using cross-validation",
    "handle class imbalance in a classification task",
    "implement attention mechanisms from scratch",
    "use transfer learning for NLP tasks",
    "preprocess text data for sentiment analysis",
    "build a recommendation system using embeddings",
    "debug vanishing gradients in deep networks",
]

from src.analysis.adversarial import _MIMICRY_TEMPLATES
matched_controls = []
for i, filler in enumerate(benign_fillers):
    template = _MIMICRY_TEMPLATES[i % len(_MIMICRY_TEMPLATES)]
    matched_controls.append(template.format(filler))

print(f'Test data:')
print(f'  Mimicry attacks:   {len(mimicry_texts)}')
print(f'  Normal prompts:    {len(normal_texts)}')
print(f'  Matched controls:  {len(matched_controls)}')
print()
print(f'Example mimicry:')
print(f'  "{mimicry_texts[0][:100]}..."')
print(f'Example matched control:')
print(f'  "{matched_controls[0][:100]}..."')

## Test 1: Does GPT-2 Small Separate Mimicry at All?

Extract raw 768-dim activations and train a linear probe. If the probe can't separate mimicry from normal even with raw activations, GPT-2 Small genuinely doesn't encode the difference.

In [ ]:
from src.model.transformer import load_model, extract_activations
from src.data.preprocessing import tokenize_prompts

# Load GPT-2 Small
gpt2_small = load_model(device)

def extract_raw_activations(model, texts, layer=0, max_length=128):
    """Extract raw residual stream activations at the last token."""
    formatted = [SYSTEM_PROMPT_TEMPLATE.format(prompt=t) for t in texts]
    all_acts = []
    batch_size = 16
    for i in range(0, len(formatted), batch_size):
        batch = formatted[i:i+batch_size]
        tokenized = tokenize_prompts(batch, max_length=max_length)
        acts = extract_activations(model, tokenized['input_ids'], tokenized['attention_mask'],
                                   layers=[layer], batch_size=batch_size)
        all_acts.append(acts[layer])
    return np.vstack(all_acts)

# Extract activations
print('Extracting GPT-2 Small activations...')
small_mimicry_acts = extract_raw_activations(gpt2_small, mimicry_texts)
small_normal_acts = extract_raw_activations(gpt2_small, normal_texts)
small_control_acts = extract_raw_activations(gpt2_small, matched_controls)

print(f'Shapes: mimicry={small_mimicry_acts.shape}, normal={small_normal_acts.shape}, control={small_control_acts.shape}')

In [ ]:
# Test 1a: Can a linear probe separate mimicry (label=1) from normal (label=0)?
X_small = np.vstack([small_mimicry_acts, small_normal_acts])
y_small = np.array([1] * len(small_mimicry_acts) + [0] * len(small_normal_acts))

clf_small = LogisticRegression(max_iter=1000, random_state=42)
scores_small = cross_val_score(clf_small, X_small, y_small, cv=5, scoring='roc_auc')

print('Test 1a: GPT-2 Small \u2014 Mimicry vs Normal (raw activations)')
print(f'  Cross-val AUC: {scores_small.mean():.3f} \u00b1 {scores_small.std():.3f}')
print(f'  (0.5 = random, 1.0 = perfect separation)')
print()

# Test 1b: Can it separate mimicry from matched controls?
# This is the harder test \u2014 same template, different topic
X_hard = np.vstack([small_mimicry_acts, small_control_acts])
y_hard = np.array([1] * len(small_mimicry_acts) + [0] * len(small_control_acts))

clf_hard = LogisticRegression(max_iter=1000, random_state=42)
scores_hard = cross_val_score(clf_hard, X_hard, y_hard, cv=min(5, len(small_control_acts)), scoring='roc_auc')

print('Test 1b: GPT-2 Small \u2014 Mimicry vs Matched Controls (raw activations)')
print(f'  Cross-val AUC: {scores_hard.mean():.3f} \u00b1 {scores_hard.std():.3f}')
print(f'  This tests whether GPT-2 Small encodes the TOPIC difference')
print(f'  (injection vs benign) when the STYLE is identical.')

## Test 2: TF-IDF on Mimicry

Does surface-level text matching catch what the neural approach misses?

In [ ]:
from src.baseline.classifiers import train_tfidf_baseline

# Train TF-IDF on full dataset
lr_pipe, _ = train_tfidf_baseline(dataset.texts, dataset.labels, seed=42)

# Test on mimicry
mimicry_tfidf_preds = lr_pipe.predict(mimicry_texts)
mimicry_tfidf_probs = lr_pipe.predict_proba(mimicry_texts)[:, 1]

# Test on matched controls (should be classified as normal)
control_tfidf_preds = lr_pipe.predict(matched_controls)

print('Test 2: TF-IDF on mimicry attacks')
print(f'  Mimicry detected:  {sum(mimicry_tfidf_preds)}/{len(mimicry_tfidf_preds)} '
      f'({sum(mimicry_tfidf_preds)/len(mimicry_tfidf_preds):.0%})')
print(f'  Mean prob:         {mimicry_tfidf_probs.mean():.3f}')
print()
print(f'  Controls falsely flagged: {sum(control_tfidf_preds)}/{len(control_tfidf_preds)}')
print()
if sum(mimicry_tfidf_preds) > sum(mimicry_tfidf_preds == 0):
    print('  TF-IDF catches mimicry \u2014 keywords like "override", "bypass" are')
    print('  detectable even in educational framing. Ensemble defense helps.')
else:
    print('  TF-IDF also misses mimicry \u2014 the educational framing dilutes')
    print('  keyword signals. This is a genuinely hard problem.')

## Test 3: GPT-2 Medium Head-to-Head

Load GPT-2 Medium (355M, d_model=1024) and repeat the linear probe test. If it separates mimicry significantly better, upgrading is worth it.

In [ ]:
# Free GPT-2 Small memory first
del gpt2_small
torch.cuda.empty_cache()

import transformer_lens

# Load GPT-2 Medium (355M params)
print('Loading GPT-2 Medium...')
gpt2_medium = transformer_lens.HookedTransformer.from_pretrained('gpt2-medium', device=device)
gpt2_medium.eval()
print(f'GPT-2 Medium: {gpt2_medium.cfg.n_layers} layers, d_model={gpt2_medium.cfg.d_model}')

if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')

In [ ]:
def extract_acts_generic(model, texts, layer=0, max_length=128):
    """Extract activations from any TransformerLens model."""
    formatted = [SYSTEM_PROMPT_TEMPLATE.format(prompt=t) for t in texts]
    hook_name = f'blocks.{layer}.hook_resid_post'
    all_acts = []
    batch_size = 8  # Smaller batch for larger model

    for i in range(0, len(formatted), batch_size):
        batch = formatted[i:i+batch_size]
        tokens = model.tokenizer(batch, padding=True, truncation=True,
                                 max_length=max_length, return_tensors='pt')
        input_ids = tokens['input_ids'].to(device)
        attn_mask = tokens['attention_mask'].to(device)

        with torch.no_grad():
            _, cache = model.run_with_cache(input_ids, names_filter=[hook_name])

        # Extract at last real token position
        last_pos = attn_mask.sum(dim=1) - 1
        batch_idx = torch.arange(input_ids.shape[0], device=device)
        acts = cache[hook_name][batch_idx, last_pos].cpu().numpy()
        all_acts.append(acts)

        del cache
        torch.cuda.empty_cache()

    return np.vstack(all_acts)

# Test at multiple layers (Medium has 24 layers \u2014 try early, mid, late)
test_layers = [0, 6, 12, 18, 23]
medium_results = {}

for layer in test_layers:
    print(f'Layer {layer}...')
    med_mimicry = extract_acts_generic(gpt2_medium, mimicry_texts, layer=layer)
    med_normal = extract_acts_generic(gpt2_medium, normal_texts, layer=layer)
    med_control = extract_acts_generic(gpt2_medium, matched_controls, layer=layer)

    # Mimicry vs Normal
    X = np.vstack([med_mimicry, med_normal])
    y = np.array([1] * len(med_mimicry) + [0] * len(med_normal))
    clf = LogisticRegression(max_iter=1000, random_state=42)
    auc_normal = cross_val_score(clf, X, y, cv=5, scoring='roc_auc').mean()

    # Mimicry vs Matched Controls (the hard test)
    X_hard = np.vstack([med_mimicry, med_control])
    y_hard = np.array([1] * len(med_mimicry) + [0] * len(med_control))
    clf_hard = LogisticRegression(max_iter=1000, random_state=42)
    n_cv = min(5, min(len(med_mimicry), len(med_control)))
    if n_cv >= 2:
        auc_control = cross_val_score(clf_hard, X_hard, y_hard, cv=n_cv, scoring='roc_auc').mean()
    else:
        auc_control = float('nan')

    medium_results[layer] = {'auc_vs_normal': auc_normal, 'auc_vs_control': auc_control}
    print(f'  AUC vs normal: {auc_normal:.3f} | AUC vs controls: {auc_control:.3f}')

## Test 4: Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Mimicry vs Normal separation
ax = axes[0]
layers = sorted(medium_results.keys())
med_aucs = [medium_results[l]['auc_vs_normal'] for l in layers]

ax.bar([-1], [scores_small.mean()], color='#D55E00', alpha=0.85, label='GPT-2 Small (L0)')
ax.bar(range(len(layers)), med_aucs, color='#0072B2', alpha=0.85, label='GPT-2 Medium')
ax.set_xticks([-1] + list(range(len(layers))))
ax.set_xticklabels(['Small\nL0'] + [f'Med\nL{l}' for l in layers], fontsize=9)
ax.set_ylabel('AUC (5-fold CV)', fontsize=11)
ax.set_title('Mimicry vs Normal: Linear Probe AUC', fontsize=12)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)

# Right: Mimicry vs Matched Controls (the hard test)
ax = axes[1]
hard_aucs = [medium_results[l]['auc_vs_control'] for l in layers]

ax.bar([-1], [scores_hard.mean()], color='#D55E00', alpha=0.85, label='GPT-2 Small (L0)')
ax.bar(range(len(layers)), hard_aucs, color='#0072B2', alpha=0.85, label='GPT-2 Medium')
ax.set_xticks([-1] + list(range(len(layers))))
ax.set_xticklabels(['Small\nL0'] + [f'Med\nL{l}' for l in layers], fontsize=9)
ax.set_ylabel('AUC (5-fold CV)', fontsize=11)
ax.set_title('Mimicry vs Matched Controls: Linear Probe AUC', fontsize=12)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)

plt.tight_layout()
save_path = DRIVE_ROOT / 'results' / 'figures' / 'mimicry_model_comparison.png'
fig.savefig(str(save_path), dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved to {save_path}')

## Decision Framework

In [ ]:
best_medium_layer = max(medium_results, key=lambda l: medium_results[l]['auc_vs_normal'])
best_medium_auc = medium_results[best_medium_layer]['auc_vs_normal']
best_medium_hard = medium_results[best_medium_layer]['auc_vs_control']
small_auc = scores_small.mean()
small_hard = scores_hard.mean()

print('=' * 60)
print('  MIMICRY DIAGNOSTIC \u2014 DECISION')
print('=' * 60)
print()
print('Mimicry vs Normal (easy test):')
print(f'  GPT-2 Small (L0):        AUC = {small_auc:.3f}')
print(f'  GPT-2 Medium (L{best_medium_layer}):     AUC = {best_medium_auc:.3f}')
print(f'  Improvement:              {best_medium_auc - small_auc:+.3f}')
print()
print('Mimicry vs Matched Controls (hard test):')
print(f'  GPT-2 Small (L0):        AUC = {small_hard:.3f}')
print(f'  GPT-2 Medium (L{best_medium_layer}):     AUC = {best_medium_hard:.3f}')
print(f'  Improvement:              {best_medium_hard - small_hard:+.3f}')
print()

delta = best_medium_auc - small_auc
delta_hard = best_medium_hard - small_hard

if delta > 0.1 or delta_hard > 0.15:
    print('RECOMMENDATION: UPGRADE')
    print(f'  GPT-2 Medium shows meaningful improvement ({delta:+.3f} / {delta_hard:+.3f}).')
    print(f'  Best layer: {best_medium_layer} (d_model={gpt2_medium.cfg.d_model})')
    print(f'  Re-running notebooks 1-12 with Medium is worth the ~1 day cost.')
    print()
    print('  To upgrade, modify src/model/transformer.py:')
    print('    model = HookedTransformer.from_pretrained("gpt2-medium", ...)')
    print(f'  And update SAE: d_input={gpt2_medium.cfg.d_model} (was 768)')
elif delta > 0.05:
    print('RECOMMENDATION: MARGINAL \u2014 CONSIDER ALTERNATIVES')
    print(f'  GPT-2 Medium shows small improvement ({delta:+.3f} / {delta_hard:+.3f}).')
    print(f'  The gain may not justify re-running 12 notebooks.')
    print('  Better to invest in ensemble defense (SAE + TF-IDF) and')
    print('  document mimicry as an honest limitation.')
else:
    print('RECOMMENDATION: DO NOT UPGRADE')
    print(f'  GPT-2 Medium shows negligible improvement ({delta:+.3f} / {delta_hard:+.3f}).')
    print('  The mimicry problem is fundamental \u2014 bigger models do not help.')
    print('  Focus on:')
    print('    1. Ensemble defense (TF-IDF catches keyword signals)')
    print('    2. Token-level features (not just last-token)')
    print('    3. Document as an honest limitation')
print()

# VRAM summary
if torch.cuda.is_available():
    print(f'VRAM used by GPT-2 Medium: {torch.cuda.memory_allocated()/1e9:.1f} GB')
    print(f'SAE on Medium would need d_input={gpt2_medium.cfg.d_model}, '
          f'd_sae={gpt2_medium.cfg.d_model * 8} (8x expansion)')

## Appendix: Per-Layer Activation Norms

Useful for choosing which layer to train the SAE on if upgrading.

In [ ]:
print('GPT-2 Medium \u2014 Per-Layer Statistics')
print(f'{"Layer":>6} {"AUC vs Norm":>12} {"AUC vs Ctrl":>12} {"Recommendation":>15}')
print('-' * 50)
for layer in sorted(medium_results.keys()):
    r = medium_results[layer]
    rec = '***' if r['auc_vs_normal'] == best_medium_auc else ''
    print(f'{layer:>6d} {r["auc_vs_normal"]:>12.3f} {r["auc_vs_control"]:>12.3f} {rec:>15}')

print()
print(f'Best layer for SAE: {best_medium_layer}')
print(f'  (highest AUC on mimicry vs normal separation)')

# Clean up
del gpt2_medium
torch.cuda.empty_cache()
print('\nGPT-2 Medium unloaded.')